# Logistic Regression & SVM (Project 2)

Run this notebook from `Project2_AML/Models` using the project virtual environment (`uv run jupyter notebook` or select the `.venv` kernel).

## Project goal
Identify up to **1,000** test customers most likely to accept the offer while keeping the number of used variables low.

## Business score
$$\text{Score} = (TP \times 10) - (FP \times 5) - (\text{NoVariables} \times 200)$$

- **TP**: contacted customer who accepted
- **FP**: contacted customer who did not accept
- **NoVariables**: number of features kept after selection

## Why we use an sklearn `Pipeline`
The helper functions in `utils/data_processing.py` are useful for exploratory notebooks, but if you apply them on the full training set **before** cross-validation, validation-fold statistics leak into feature selection.

Our LR/SVM code keeps the same preprocessing order as the team notebook:

1. variance filter (`threshold=0.01`)
2. correlation filter (`threshold=0.9`)
3. standardization
4. `SelectFromModel` (L1 selector)
5. final classifier (`LogisticRegression` or `SVC`)

Everything above stays **inside one pipeline**, so each CV fold refits preprocessing on training data only.

## Fitting workflow
1. **Tune k** (50 to 1000): cross-validated business score for different contact-list sizes
2. **Tune model hyperparameters** (`C`, and `gamma` for SVM) with `GridSearchCV` using the best k
3. **Fit final model** on all training data and rank test customers by confidence
4. **Save outputs**: selected customer indices, selected variables, rankings, and the fitted pipeline

In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd

# Make imports work whether the notebook is opened from `Models/` or `Project2_AML/`.
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "utils" / "data_processing.py").exists():
    MODELS_DIR = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / "Models" / "utils" / "data_processing.py").exists():
    MODELS_DIR = NOTEBOOK_DIR / "Models"
else:
    raise FileNotFoundError(
        "Open this notebook from Project2_AML/Models so utils/ can be imported."
    )

if str(MODELS_DIR) not in sys.path:
    sys.path.insert(0, str(MODELS_DIR))

from utils.business_scoring_sanity_checks import (
    test_score_from_mask_and_indices_agree,
    test_top_k_policy_picks_highest_scores,
    test_scorer_works_with_logistic_regression,
)
from logistic_regression_business_experiment import run_experiment as run_logistic_regression
from svm_business_experiment import run_experiment as run_svm

print("Notebook path:", MODELS_DIR)
print("Sanity checks passed.")
test_score_from_mask_and_indices_agree()
test_top_k_policy_picks_highest_scores()
test_scorer_works_with_logistic_regression()

## Run Logistic Regression

This cell runs the full experiment:
- k-search with 5-fold CV (about **5 minutes** on this dataset)
- grid search over selector and model `C`
- test-set ranking and output files in `outputs/logistic_regression/`

If you already ran the experiment, you can reload the saved summary instead of refitting:

```python
import json
from pathlib import Path
lr_summary = json.loads(Path("outputs/logistic_regression/summary.json").read_text())
```

In [ ]:
lr_summary = run_logistic_regression()
pd.Series(lr_summary)

## Run SVM

Same business-score workflow as Logistic Regression, but the final classifier is an RBF `SVC`.

**Note:** SVM is much slower than Logistic Regression on 5,000 rows. The notebook uses `quick=True` for a faster verification run. For final submission, run the full search from terminal:

```bash
uv run python svm_business_experiment.py
```

Outputs are written to `outputs/svm/`.

In [ ]:
svm_summary = run_svm(quick=True)
pd.Series(svm_summary)

## Compare models

Use cross-validated business score and the number of retained variables to choose the model to submit.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "model": "logistic_regression",
            "best_k": lr_summary["best_k"],
            "best_cv_business_score": lr_summary["best_cv_business_score"],
            "n_variables": lr_summary["n_variables"],
            "selected_customers": lr_summary["selected_customers"],
            "output_dir": lr_summary["output_dir"],
        },
        {
            "model": "svm",
            "best_k": svm_summary["best_k"],
            "best_cv_business_score": svm_summary["best_cv_business_score"],
            "n_variables": svm_summary["n_variables"],
            "selected_customers": svm_summary["selected_customers"],
            "output_dir": svm_summary["output_dir"],
        },
    ]
).sort_values("best_cv_business_score", ascending=False)

comparison